# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

# Automated

In [52]:
import polars as pl

In [53]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [54]:
import os
import shutil

# Remove all files in subfolders edgotyping_profiles
profile_dirs = [
    # f"{outputs_dir}/edgotype_profiles/manual",
    f"{outputs_dir}/edgotype_profiles/auto_norm", 
    f"{outputs_dir}/edgotype_profiles/auto_summed"
]

for profile_dir in profile_dirs:
    if os.path.exists(profile_dir):
        # Remove all files in the directory
        for filename in os.listdir(profile_dir):
            file_path = os.path.join(profile_dir, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')
        print(f"Cleared all files from {profile_dir}")
    else:
        print(f"Directory {profile_dir} does not exist")

Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_norm
Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_summed


In [55]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["summed_consensus_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,summed_consensus_score,wt_summed_score,condition_name,assay_id,ppi_group,seq_ad,seq_db,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,str,i64,str,i64,i64,str,str,str,str,str,str,str,f64,f64,str
4761,2713,4554,"""VUSAPWT1B1""",412.840528,5884.526979,"""LWH1""",1,"""4761_2713_4554_VUSAPWT1B1""",1,1,"""AP2B1""","""MLH1""","""Ala586Asp""","""1757C>A""","""ENSG00000076242""","""RC4""","""1_Pathogenic""",null,2.0,"""MLH1_Ala586Asp"""
4759,5066,201831,"""VUSAPWT1B2""",1805.854932,2578.019667,"""LWH1""",1,"""4759_5066_201831_VUSAPWT1B2""",1,1,"""VPS37B""","""BAG3""","""Leu462Pro""","""1385T>C""","""ENSG00000151929""","""CEGS2""","""1_Pathogenic""",null,1.0,"""BAG3_Leu462Pro"""
6490,2920,214422,"""VUSAPWT2B1""",4227.108633,4941.943946,"""LWA""",2,"""6490_2920_214422_VUSAPWT2B1""",1,1,"""PIAS2""","""GFAP""","""Arg239Leu""","""716G>T""","""ENSG00000131095""","""CEGS2""","""1_Pathogenic""",null,2.0,"""GFAP_Arg239Leu"""
439,52920,2667,"""VUSAPWT6B1""",3015.88244,669.847077,"""LWH1""",6,"""439_52920_2667_VUSAPWT6B1""",1,1,"""RNF208""","""LITAF""","""Pro135Arg""","""404C>G""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,null,"""LITAF_Pro135Arg"""
56215,52920,2675,"""VUSAPWT1B1""",8133.114039,8432.32835,"""LWH25""",1,"""56215_52920_2675_VUSAPWT1B1""",1,1,"""HIP1""","""LITAF""","""Trp116Gly""","""346T>G""","""ENSG00000189067""","""Edgotyping3""","""1_Pathogenic""",null,1.0,"""LITAF_Trp116Gly"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4761,2713,200856,"""VUSAPWT1B1""",3656.96612,5884.526979,"""LWH10""",1,"""4761_2713_200856_VUSAPWT1B1""",1,1,"""AP2B1""","""MLH1""","""Asp304Gly""","""911A>G""","""ENSG00000076242""","""CEGS2""","""3_Conflicting""",6.8405e-7,1.0,"""MLH1_Asp304Gly"""
11288,3180,214556,"""VUSAPWT1B2""",4173.615793,2379.97782,"""LWH1""",1,"""11288_3180_214556_VUSAPWT1B2""",1,1,"""CREBRF""","""NAA10""","""Arg116Trp""","""346C>T""","""ENSG00000102030""","""CEGS2""","""1_Pathogenic""",null,2.0,"""NAA10_Arg116Trp"""
71163,52920,2681,"""VUSAPWT2B1""",1163.810479,4989.452432,"""LWH1""",2,"""71163_52920_2681_VUSAPWT2B1""",1,1,"""KCNIP1""","""LITAF""","""Pro84Leu""","""251C>T""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,2.0,"""LITAF_Pro84Leu"""


In [56]:
scores_pd = scores.to_pandas()
scores_pd[
    (scores_pd['ad_orf_id'] == 14891) &
    (scores_pd['db_mut_id'] == 213584)
]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,summed_consensus_score,wt_summed_score,condition_name,assay_id,ppi_group,seq_ad,...,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
402,14891,143,213584,VUSAPWT1B2,3617.949132,5779.031134,LWH1,1,14891_143_213584_VUSAPWT1B2,1,...,EED,EZH2,Arg684Cys,2050C>T,ENSG00000106462,CEGS2,1_Pathogenic,NaN,2.0,EZH2_Arg684Cys


In [57]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    duplicates = gene_scores.group_by(["ad_orf_id", "db_mut_id"]).len().filter(pl.col("len") > 1)
    profile = gene_scores.pivot(
                values="summed_consensus_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)

In [58]:
# ## Add a perturbation category for each edge to use for the N2H validation

# # Extract WT scores
# wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# # Merge WT scores back onto the main df
# scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# # Compute allele/WT ratio
# scores_with_wt = scores_with_wt.with_columns(
#     (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
# )

# # Determine if the allele is perturbed - binary output
# threshold = 0.5
# scores_with_wt = scores_with_wt.with_columns(
#     ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
# )

# # Build perturbation_classes expression
# bins = [
#     (1.0, float("inf"), "increased edge"),
#     (0.8, 1.0, "similar"),
#     (0.6, 0.8, "slightly perturbed"),
#     (0.4, 0.6, "perturbed"),
#     (0.2, 0.4, "highly perturbed"),
#     (0.0, 0.2, "very highly perturbed"),
# ]

# perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

# perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

# perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

# perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

# for lower, upper, label in bins:
#     perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


# perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




# scores_with_wt = scores_with_wt.with_columns(
#     perturbation_expr.alias("perturbation_classes")
# )

# scores_with_perturbed = scores_with_wt.select("*")


# pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

# pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

# pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

# pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

# bins_2 = [
#     (float("inf"), 1.0, "increased edge"),
#     (1, 0.7, 'slightly perturbed'),
#     (0.7, 0.3, 'perturbed'),
#     (0.3, 0, 'highly perturbed')
# ]

# for upper, lower, label in bins_2:
#     pert2 = pert2.when(
#         (pl.col("ratio") <= upper) &
#         (pl.col("ratio") >  lower)
#     ).then(pl.lit(label))

# pert2 = pert2.otherwise(pl.lit("unclassified"))

# scores_with_perturbed = scores_with_wt.with_columns(
#     perturbation_classes_2 = pert2.alias("perturbation_classes_2")
# )



# cols_to_keep = ['ad_orf_id',
#  'db_orf_id',
#  'db_mut_id',
#  'normalized_score',
#  'ad_symbol',
#  'symbol',
#  'clinvar_clnsig_clean',
#  'normalized_score_wt',
#  'ratio',
#  'perturbed',
#  'perturbation_classes',
#  "perturbation_classes_2"]

# scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

# scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")